In [ ]:
import glob, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

# ── Load all daily draw CSVs ──────────────────────────────────────────────────
files = glob.glob('daily_draws/*.csv')
daily_raw = pd.concat([
    pd.read_csv(f).assign(draw_date=os.path.splitext(os.path.basename(f))[0])
    for f in files
])
daily_raw.rename(columns={'time [UTC]': 'date', 'value': 'daily'}, inplace=True)
daily_raw['date']      = pd.to_datetime(daily_raw['date'],      format='mixed').dt.date
daily_raw['draw_date'] = pd.to_datetime(
    daily_raw['draw_date'].str.replace('draw_', ''), format='mixed'
).dt.date
daily_pivot     = daily_raw.pivot_table(index='date', columns='draw_date',
                                        values='daily', aggfunc='first')
daily_draw_cols = daily_pivot.columns.tolist()

# ── Weekly benchmarks ────────────────────────────────────────────────────────
weekly_final = pivot[draw_cols].mean(axis=1)
weekly_final = weekly_final / weekly_final.max() * 100

# ── Drop partial-week benchmarks ─────────────────────────────────────────────
daily_start = pd.to_datetime(daily_pivot.index.min())
daily_end   = pd.to_datetime(daily_pivot.index.max())
weekly_final = weekly_final[
    (pd.to_datetime(weekly_final.index) >= daily_start) &
    (pd.to_datetime(weekly_final.index) + pd.Timedelta(days=6) <= daily_end)
]
print(f"Benchmarks after trimming partial weeks: {len(weekly_final)}")
print(f"  First: {pd.to_datetime(weekly_final.index[0]).date()}  "
      f"Last:  {pd.to_datetime(weekly_final.index[-1]).date()}")

# ── Scale reconciliation ─────────────────────────────────────────────────────
def estimate_scale_constant(indicator: pd.Series, benchmarks: pd.Series) -> float:
    dates = pd.to_datetime(indicator.index)
    weekly_means = []
    for wk in pd.to_datetime(benchmarks.index):
        mask = (dates >= wk) & (dates <= wk + pd.Timedelta(days=6))
        vals = indicator[mask]
        weekly_means.append(vals.mean() if len(vals) > 0 else np.nan)
    weekly_means = pd.Series(weekly_means, index=benchmarks.index)
    valid = weekly_means.notna()
    return float(benchmarks[valid].mean() / weekly_means[valid].mean())


# ── TRUE Denton proportional adjustment ──────────────────────────────────────
# min  Σ_{t=2}^T (X_t/x_t - X_{t-1}/x_{t-1})²
# s.t. (1/n_k) Σ_{t∈w_k} X_t = B_k   for every benchmark week k
#
# KKT solution:  p* = p₀ + Q_reg⁻¹ A' (A Q_reg⁻¹ A')⁻¹ (b_sum − A p₀)
#
# FIX: replace np.linalg.pinv(Q) with Tikhonov Q_reg = Q + ε·I solved via
# scipy.linalg.solve. pinv on a large near-singular matrix is numerically
# unstable and produces a huge p_opt at the boundary (t=0), which becomes
# the max after normalisation and compresses the entire series to near-zero.
def denton_proportional(indicator: pd.Series, benchmarks: pd.Series) -> pd.Series:
    x, n  = indicator.values.astype(float), len(indicator)
    dates = pd.to_datetime(indicator.index)
    m     = len(benchmarks)

    J, n_k_arr = np.zeros((m, n)), np.zeros(m)
    for k, wk in enumerate(pd.to_datetime(benchmarks.index)):
        mask       = (dates >= wk) & (dates <= wk + pd.Timedelta(days=6))
        n_k_arr[k] = mask.sum()
        J[k, mask] = 1.0

    A     = J * x[np.newaxis, :]                       # (m, n)
    b_sum = benchmarks.values.astype(float) * n_k_arr  # (m,)

    # First-difference penalty Q = D'D  (tridiagonal, rank n-1)
    D     = np.diff(np.eye(n), axis=0)
    Q     = D.T @ D

    # Tikhonov regularisation: Q + ε·I  → full rank, stable inversion
    # ε is tiny relative to Q's diagonal (≈1-2), so it barely changes the solution
    # but eliminates the null-space instability that pinv mishandles for large n
    eps   = 1e-6
    Q_reg = Q + eps * np.eye(n)

    p0 = np.ones(n)

    # Solve Q_reg @ V = A'  →  V = Q_reg⁻¹ A'  (n×m)
    # scipy.linalg.solve uses LU/Cholesky: far more stable than forming pinv
    V   = linalg.solve(Q_reg, A.T, assume_a='pos')  # (n, m)
    M   = A @ V                                       # (m, m)
    lam = linalg.solve(M, b_sum - A @ p0)            # (m,)

    p_opt = p0 + V @ lam
    return pd.Series(p_opt * x, index=indicator.index)


# ── Method 1: Naive average + Denton ─────────────────────────────────────────
naive_avg    = daily_pivot[daily_draw_cols].mean(axis=1)
C_naive      = estimate_scale_constant(naive_avg, weekly_final)
print(f"\nScale constant (naive avg):    C = {C_naive:.4f}")
naive_denton = denton_proportional(naive_avg * C_naive, weekly_final)
naive_denton = naive_denton / naive_denton.max() * 100


# ── Method 2: Rescale to consensus peak, then average + Denton ───────────────
daily_peaks          = daily_pivot[daily_draw_cols].idxmax()
consensus_peak_daily = daily_peaks.mode()[0]
n_agree = (daily_peaks == consensus_peak_daily).sum()
print(f"Daily consensus peak: {consensus_peak_daily}  ({n_agree}/{len(daily_draw_cols)} draws)")

daily_rescaled = daily_pivot[daily_draw_cols].copy().astype(float)
for col in daily_draw_cols:
    if daily_peaks[col] != consensus_peak_daily:
        val = daily_rescaled.loc[consensus_peak_daily, col]
        if val > 0:
            daily_rescaled[col] *= 100.0 / val
            print(f"  Rescaled {col} (peaked at {daily_peaks[col]}, had {val:.1f} at consensus)")

rescaled_avg    = daily_rescaled.mean(axis=1)
C_rescaled      = estimate_scale_constant(rescaled_avg, weekly_final)
print(f"Scale constant (rescaled avg): C = {C_rescaled:.4f}")
rescaled_denton = denton_proportional(rescaled_avg * C_rescaled, weekly_final)
rescaled_denton = rescaled_denton / rescaled_denton.max() * 100


# ── Sanity check: within-week mean of Denton output should ≈ benchmark ───────
mid_idx  = len(weekly_final) // 2
wk_check = pd.to_datetime(weekly_final.index[mid_idx])
mask_chk = (pd.to_datetime(naive_denton.index) >= wk_check) & \
           (pd.to_datetime(naive_denton.index) <= wk_check + pd.Timedelta(days=6))
print(f"\n── Sanity check (week starting {wk_check.date()}) ──")
print(f"  naive_denton mean in week: {naive_denton[mask_chk].mean():.6f}")
print(f"  weekly_final benchmark:    {weekly_final.iloc[mid_idx]:.6f}")
print(f"  n days in window:          {mask_chk.sum()}")


# ── Plot comparison ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(naive_denton.index,    naive_denton.values,
        color='red',      lw=1.5, ls='--', label='Naive avg + Denton')
ax.plot(rescaled_denton.index, rescaled_denton.values,
        color='darkblue', lw=2,            label='Rescaled avg + Denton')
ax.set(xlabel='Date', ylabel='Index (max = 100)',
       title=f'Daily Draws: Both Denton-adjusted ({len(daily_draw_cols)} draws)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

diff = (rescaled_denton - naive_denton).abs()
print(f"\nMax absolute difference:  {diff.max():.2f}")
print(f"Mean absolute difference: {diff.mean():.2f}")

In [ ]:
# ── Diagnose: are benchmark dates week-END or week-START? ────────────────────
bench_dates = pd.to_datetime(weekly_final.index)
daily_dates = pd.to_datetime(daily_pivot.index)

print("=== Sample benchmark dates ===")
for d in bench_dates[:5]:
    print(f"  {d.date()}  →  {d.day_name()}")

print(f"\nWeekday distribution across all {len(bench_dates)} benchmark dates:")
print(bench_dates.day_name().value_counts().sort_index())

print(f"\nDaily data range:     {daily_dates.min().date()} → {daily_dates.max().date()}")
print(f"Benchmark date range: {bench_dates.min().date()} → {bench_dates.max().date()}")

# For each benchmark date test both interpretations
print("\n=== Coverage check (first 5 benchmarks) ===")
print(f"{'Date':<14} {'Weekday':<12} "
      f"{'END window (d-6..d)':<23} {'START window (d..d+6)'}")
for d in bench_dates[:5]:
    n_end   = ((daily_dates >= d - pd.Timedelta(days=6)) & (daily_dates <= d)).sum()
    n_start = ((daily_dates >= d) & (daily_dates <= d + pd.Timedelta(days=6))).sum()
    print(f"  {str(d.date()):<12}  {d.day_name():<12}  {n_end:>3} daily obs"
          f"                 {n_start:>3} daily obs")

# Aggregate over all benchmarks
end_total   = sum(((daily_dates >= d - pd.Timedelta(days=6)) & (daily_dates <= d)).sum()
                  for d in bench_dates)
start_total = sum(((daily_dates >= d) & (daily_dates <= d + pd.Timedelta(days=6))).sum()
                  for d in bench_dates)

print(f"\nTotal daily obs matched:")
print(f"  Treating dates as week-END   (d-6 .. d  ): {end_total}")
print(f"  Treating dates as week-START (d   .. d+6): {start_total}")

if start_total >= end_total:
    print(f"\n→ Verdict: week-START (d .. d+6)  ✓")
    print(f"  GT Sunday label = first day of the window (Sun→Sat).")
    print(f"  Denton mask [wk, wk+6] is CORRECT.")
    print(f"  GDELT must use resample('W-SAT') shifted -6 days to match.")
else:
    print(f"\n→ Verdict: week-END (d-6 .. d)  ✓")
    print(f"  GT Sunday label = last day of the window (Mon→Sun).")
    print(f"  Denton mask must be changed to [wk-6, wk].")
    print(f"  GDELT resample('W') is CORRECT.")

# Data Science Tools and Ecosystem


In this notebook, Data Science Tools and Ecosystem are summarized.

**Objectives:**

- List popular languages for Data Science.
- Introduce commonly used libraries in Data Science.
- Explore examples of evaluating arithmetic expressions in Python.
- Understand how to convert minutes to hours using Python.
- Summarize the Data Science tools and ecosystem.


Some of the popular languages that Data Scientists use are:

1. Python
2. R
3. SQL
4. Julia
5. Scala


Some of the commonly used libraries by Data Scientists include:

1. NumPy
2. Pandas
3. Matplotlib
4. Scikit-Learn
5. TensorFlow


| Data Science Tools    |
|-----------------------|
| Jupyter Notebook      |
| RStudio               |
| Visual Studio Code    |


### Examples of Evaluating Arithmetic Expressions in Python

In Python, you can perform various arithmetic operations. Here are some examples:

1. **Addition:**
   ```python
   result = 5 + 3
   # result will be 8
result = 10 - 4
# result will be 6


In [6]:
# This a simple arithmetic expression to mutiply then add integers
(3*4)+5

17

In [8]:
#This will convert 200 minutes to hours by diving by 60
200/60

3.3333333333333335

## Author
Andrea Lamacchia


## Robustness Check: Denton on Naive Average vs Denton on Rescaled Average

**Working assumption:** weekly Gtrends = within-week **mean** of the underlying daily query share × an unknown multiplicative constant.  
The constant is estimated by aligning the two series on the full-sample mean (scale reconciliation) before applying the Denton proportional adjustment.  
Pro-rata via weekly **sum** (as often coded by default) is wrong here and throws away cross-week information.

Two methods compared:
1. **Naive avg + Denton** — average the raw daily draws, reconcile scale, apply Denton.
2. **Rescaled avg + Denton** — rescale each draw to a consensus peak first, then average, reconcile scale, apply Denton.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

NICE = {
    'gtrends':       'Google Trends',
    'neg_art_count': 'Neg. article count',
    'avg_tone':      'Avg tone (weighted)'
}

# Work on clean (no NaN) weekly series
W = weekly_all.dropna().copy()
print(f'Clean weekly obs: {len(W)}  ({W.index[0].date()} → {W.index[-1].date()})')
W.describe().round(3)

In [ ]:
# ── Visual inspection ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
colors = ['#2196F3', '#FF5722', '#4CAF50']

for ax, col, color in zip(axes, W.columns, colors):
    ax.plot(W.index, W[col], color=color, linewidth=1.2)
    ax.set_ylabel(NICE[col], fontsize=10)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Clean weekly series', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Week-end (Saturday)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Helper function ───────────────────────────────────────────────────────────
PAIRS = [
    ('gtrends',       'neg_art_count', 'GT vs Neg.articles'),
    ('gtrends',       'avg_tone',      'GT vs Tone'),
    ('neg_art_count', 'avg_tone',      'Neg.articles vs Tone'),
]

def corr_table(data, pairs, label=''):
    rows = []
    for xc, yc, name in pairs:
        x, y = data[xc].dropna(), data[yc].dropna()
        idx  = x.index.intersection(y.index)
        x, y = x[idx], y[idx]
        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)
        rows.append({'Pair': name, 'n': len(x),
                     'Pearson r': round(pr, 4), 'Pearson p': round(pp, 4),
                     'Spearman ρ': round(sr, 4), 'Spearman p': round(sp, 4)})
    tbl = pd.DataFrame(rows).set_index('Pair')
    print(f'=== {label} ===')
    return tbl

# ── Correlations in LEVELS ────────────────────────────────────────────────────
corr_lev = corr_table(W, PAIRS, 'Correlations IN LEVELS')
display(corr_lev)

# ── Correlations in FIRST DIFFERENCES ────────────────────────────────────────
W_diff = W.diff().dropna()

PAIRS_D = [
    ('gtrends',       'neg_art_count', 'ΔGT vs ΔNeg.articles'),
    ('gtrends',       'avg_tone',      'ΔGT vs ΔTone'),
    ('neg_art_count', 'avg_tone',      'ΔNeg.articles vs ΔTone'),
]
corr_dif = corr_table(W_diff, PAIRS_D, 'Correlations IN FIRST DIFFERENCES')
display(corr_dif)

In [ ]:
# ── Heatmaps: levels vs differences ──────────────────────────────────────────
Wr  = W.rename(columns=NICE)
Wdr = W_diff.rename(columns=NICE)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
configs = [
    (Wr,  'pearson',  'Pearson — levels'),
    (Wr,  'spearman', 'Spearman — levels'),
    (Wdr, 'pearson',  'Pearson — first differences'),
    (Wdr, 'spearman', 'Spearman — first differences'),
]
for ax, (data, method, title) in zip(axes.flat, configs):
    mat = data.corr(method=method)
    sns.heatmap(mat, ax=ax, annot=True, fmt='.3f', cmap='RdBu_r',
                vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 12})
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Correlation matrices — levels vs first differences', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Clean daily series ────────────────────────────────────────────────────────
D = daily_all.dropna().copy()
print(f'Clean daily obs: {len(D)}  ({D.index[0].date()} → {D.index[-1].date()})')
D.describe().round(3)

In [ ]:
# ── Visual inspection ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
colors = ['#2196F3', '#FF5722', '#4CAF50']

for ax, col, color in zip(axes, D.columns, colors):
    ax.plot(D.index, D[col], color=color, linewidth=1.2)
    ax.set_ylabel(NICE[col], fontsize=10)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Clean daily series', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# ── Helper pairs for daily ────────────────────────────────────────────────────
PAIRS_DAILY = [
    ('gtrends',       'neg_art_count', 'GT vs Neg.articles'),
    ('gtrends',       'avg_tone',      'GT vs Tone'),
    ('neg_art_count', 'avg_tone',      'Neg.articles vs Tone'),
]

# ── Correlations in LEVELS ────────────────────────────────────────────────────
corr_lev_d = corr_table(D, PAIRS_DAILY, 'Daily correlations IN LEVELS')
display(corr_lev_d)

# ── Correlations in FIRST DIFFERENCES ────────────────────────────────────────
D_diff = D.diff().dropna()

PAIRS_DAILY_D = [
    ('gtrends',       'neg_art_count', 'ΔGT vs ΔNeg.articles'),
    ('gtrends',       'avg_tone',      'ΔGT vs ΔTone'),
    ('neg_art_count', 'avg_tone',      'ΔNeg.articles vs ΔTone'),
]
corr_dif_d = corr_table(D_diff, PAIRS_DAILY_D, 'Daily correlations IN FIRST DIFFERENCES')
display(corr_dif_d)

In [ ]:
# ── Heatmaps: daily levels vs differences ─────────────────────────────────────
Dr  = D.rename(columns=NICE)
Ddr = D_diff.rename(columns=NICE)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
configs_d = [
    (Dr,  'pearson',  'Pearson — daily levels'),
    (Dr,  'spearman', 'Spearman — daily levels'),
    (Ddr, 'pearson',  'Pearson — daily first differences'),
    (Ddr, 'spearman', 'Spearman — daily first differences'),
]
for ax, (data, method, title) in zip(axes.flat, configs_d):
    mat = data.corr(method=method)
    sns.heatmap(mat, ax=ax, annot=True, fmt='.3f', cmap='RdBu_r',
                vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 12})
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Daily — Correlation matrices: levels vs first differences', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('dailycorrelationsmatrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Multicollinearity check: neg_art_count vs avg_tone ───────────────────────
THRESHOLD = 0.70

def multicol_check(data, xc, yc, label):
    x, y = data[xc].dropna(), data[yc].dropna()
    idx  = x.index.intersection(y.index)
    pr, _ = stats.pearsonr(x[idx], y[idx])
    sr, _ = stats.spearmanr(x[idx], y[idx])
    flag = '⚠️  HIGH' if max(abs(pr), abs(sr)) > THRESHOLD else '✓  OK'
    print(f'{label:25s}  Pearson r={pr:+.4f}  Spearman ρ={sr:+.4f}  → {flag}')

print('─── Multicollinearity: Neg.article counts vs Avg tone ───')
multicol_check(W,      'neg_art_count', 'avg_tone', 'Levels')
multicol_check(W_diff, 'neg_art_count', 'avg_tone', 'First differences')

In [ ]:
# ── Scatter plots — levels ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (xc, yc, _) in zip(axes, PAIRS):
    x, y = W[xc], W[yc]
    ax.scatter(x, y, alpha=0.4, s=15, color='steelblue')
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m*xs + b, color='crimson', lw=1.5)
    pr, _ = stats.pearsonr(x, y)
    sr, _ = stats.spearmanr(x, y)
    ax.set_xlabel(NICE[xc]); ax.set_ylabel(NICE[yc])
    ax.set_title(f'r = {pr:.3f}   ρ = {sr:.3f}', fontsize=10)
    ax.grid(True, alpha=0.3)
plt.suptitle('Scatter plots — levels', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Scatter plots — first differences ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (xc, yc, _) in zip(axes, PAIRS_D):
    x, y = W_diff[xc], W_diff[yc]
    ax.scatter(x, y, alpha=0.4, s=15, color='darkorange')
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m*xs + b, color='navy', lw=1.5)
    pr, _ = stats.pearsonr(x, y)
    sr, _ = stats.spearmanr(x, y)
    ax.set_xlabel(NICE[xc]); ax.set_ylabel(NICE[yc])
    ax.set_title(f'Δ: r = {pr:.3f}   ρ = {sr:.3f}', fontsize=10)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.grid(True, alpha=0.3)
plt.suptitle('Scatter plots — first differences', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.tsa.stattools import ccf as sm_ccf, grangercausalitytests
from statsmodels.tsa.api import VAR

def plot_ccf(x, y, x_label, y_label, max_lags=16, title='', ax=None):
    def _ccf(a, b, nlags):
        try:
            result = sm_ccf(a, b, nlags=nlags, alpha=None)
        except TypeError:
            result = sm_ccf(a, b, unbiased=False, alpha=None)
        if isinstance(result, tuple):
            result = result[0]
        arr = np.asarray(result, dtype=float)
        return arr[:nlags + 1]   # cap at nlags+1 values

    pos_ccf = _ccf(x, y, max_lags)   # lags 0..k  (x leads y)
    neg_ccf = _ccf(y, x, max_lags)   # lags 0..k  (y leads x)

    # Build two-sided CCF — derive lags from actual array lengths
    neg_part = neg_ccf[1:][::-1]          # lags -(k), ..., -1
    pos_part = pos_ccf                     # lags 0, 1, ..., k
    vals = np.concatenate([neg_part, pos_part])
    lags = np.arange(-len(neg_part), len(pos_part))   # always matches vals

    n  = len(x)
    ci = 1.96 / np.sqrt(n)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))

    bar_colors = ['steelblue' if abs(v) > ci else 'lightsteelblue' for v in vals]
    ax.bar(lags, vals, color=bar_colors, width=0.8)
    ax.axhline( ci, color='crimson', lw=1, ls='--', label='95% CI')
    ax.axhline(-ci, color='crimson', lw=1, ls='--')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel(f'Lag (+ = {x_label} leads {y_label})')
    ax.set_ylabel('Cross-correlation')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

def var_lag_selection(data, cols, max_lags=8, label=''):
    sub = data[cols].dropna()
    res = VAR(sub).select_order(maxlags=max_lags)
    print(f'─── {label} ───  AIC={res.aic}  BIC={res.bic}')
    return max(res.aic, res.bic, 1)

def granger_summary(data, cause_col, effect_col, max_lag, cause_label, effect_label):
    xy  = data[[effect_col, cause_col]].dropna()
    res = grangercausalitytests(xy, maxlag=max_lag, verbose=False)
    F, p, *_ = res[max_lag][0]['ssr_ftest']
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    return {'Cause → Effect': f'{cause_label} → {effect_label}',
            'Lag': max_lag, 'F-stat': round(F,4), 'p-value': round(p,4), 'Sig.': sig}

def granger_pvals(data, cause_col, effect_col, max_lag):
    xy  = data[[effect_col, cause_col]].dropna()
    res = grangercausalitytests(xy, maxlag=max_lag, verbose=False)
    return [res[k][0]['ssr_ftest'][1] for k in range(1, max_lag+1)]

print('Helper functions ready.')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_ccf(W_diff['neg_art_count'].values, W_diff['gtrends'].values,
         'ΔArticles', 'ΔGT', max_lags=12,
         title='Weekly CCF — ΔNeg.articles vs ΔGoogle Trends\n(+k: Articles lead GT by k weeks)',
         ax=axes[0])
plot_ccf(W_diff['avg_tone'].values, W_diff['gtrends'].values,
         'ΔTone', 'ΔGT', max_lags=12,
         title='Weekly CCF — ΔTone vs ΔGoogle Trends\n(+k: Tone leads GT by k weeks)',
         ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# ── VAR lag selection (defines lag_w_art and lag_w_tone) ─────────────────────
lag_w_art  = var_lag_selection(W_diff, ['neg_art_count', 'gtrends'],
                               label='Articles ↔ GT')
lag_w_tone = var_lag_selection(W_diff, ['avg_tone', 'gtrends'],
                               label='Tone ↔ GT')

# ── Granger causality summary table ──────────────────────────────────────────
rows_w = [
    granger_summary(W_diff, 'neg_art_count', 'gtrends',       lag_w_art,  'ΔArticles', 'ΔGT'),
    granger_summary(W_diff, 'gtrends',       'neg_art_count', lag_w_art,  'ΔGT',       'ΔArticles'),
    granger_summary(W_diff, 'avg_tone',      'gtrends',       lag_w_tone, 'ΔTone',     'ΔGT'),
    granger_summary(W_diff, 'gtrends',       'avg_tone',      lag_w_tone, 'ΔGT',       'ΔTone'),
]
gc_weekly = pd.DataFrame(rows_w).set_index('Cause → Effect')
print('=== Weekly Granger causality (*** p<0.001  ** p<0.01  * p<0.05) ===')
display(gc_weekly)

In [ ]:
MAX_LAG_W = 8
combos_w = [
    ('neg_art_count', 'gtrends',       'ΔArticles → ΔGT'),
    ('gtrends',       'neg_art_count', 'ΔGT → ΔArticles'),
    ('avg_tone',      'gtrends',       'ΔTone → ΔGT'),
    ('gtrends',       'avg_tone',      'ΔGT → ΔTone'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharey=True)
for ax, (cause, effect, title) in zip(axes.flat, combos_w):
    pvals = granger_pvals(W_diff, cause, effect, MAX_LAG_W)
    ax.bar(range(1, MAX_LAG_W+1), pvals,
           color=['crimson' if p < 0.05 else 'steelblue' for p in pvals])
    ax.axhline(0.05, color='black', lw=1, ls='--', label='p=0.05')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('p-value')
    ax.set_xticks(range(1, MAX_LAG_W+1))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Weekly — Granger causality p-values by lag\n(red = significant at 5%)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_ccf(D_diff['neg_art_count'].values, D_diff['gtrends'].values,
         'ΔArticles', 'ΔGT', max_lags=30,
         title='Daily CCF — ΔNeg.articles vs ΔGoogle Trends\n(+k: Articles lead GT by k days)',
         ax=axes[0])
plot_ccf(D_diff['avg_tone'].values, D_diff['gtrends'].values,
         'ΔTone', 'ΔGT', max_lags=30,
         title='Daily CCF — ΔTone vs ΔGoogle Trends\n(+k: Tone leads GT by k days)',
         ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# ── VAR lag selection (defines lag_d_art and lag_d_tone) ─────────────────────
lag_d_art  = var_lag_selection(D_diff, ['neg_art_count', 'gtrends'],
                               label='Articles ↔ GT (daily)')
lag_d_tone = var_lag_selection(D_diff, ['avg_tone', 'gtrends'],
                               label='Tone ↔ GT (daily)')

# ── Granger causality summary table ──────────────────────────────────────────
rows_d = [
    granger_summary(D_diff, 'neg_art_count', 'gtrends',       lag_d_art,  'ΔArticles', 'ΔGT'),
    granger_summary(D_diff, 'gtrends',       'neg_art_count', lag_d_art,  'ΔGT',       'ΔArticles'),
    granger_summary(D_diff, 'avg_tone',      'gtrends',       lag_d_tone, 'ΔTone',     'ΔGT'),
    granger_summary(D_diff, 'gtrends',       'avg_tone',      lag_d_tone, 'ΔGT',       'ΔTone'),
]
gc_daily = pd.DataFrame(rows_d).set_index('Cause → Effect')
print('=== Daily Granger causality (*** p<0.001  ** p<0.01  * p<0.05) ===')
display(gc_daily)

In [ ]:
MAX_LAG_D = lag_d_art  # use the data-driven lag for x-axis extent
combos_d = [
    ('neg_art_count', 'gtrends',       'ΔArticles → ΔGT'),
    ('gtrends',       'neg_art_count', 'ΔGT → ΔArticles'),
    ('avg_tone',      'gtrends',       'ΔTone → ΔGT'),
    ('gtrends',       'avg_tone',      'ΔGT → ΔTone'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharey=True)
for ax, (cause, effect, title) in zip(axes.flat, combos_d):
    pvals = granger_pvals(D_diff, cause, effect, MAX_LAG_D)
    ax.bar(range(1, MAX_LAG_D+1), pvals,
           color=['crimson' if p < 0.05 else 'steelblue' for p in pvals], width=0.8)
    ax.axhline(0.05, color='black', lw=1, ls='--', label='p=0.05')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('p-value')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Daily — Granger causality p-values by lag\n(red = significant at 5%)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()